# Test Task: Dota 2 First-Pick Prediction

## Background

In Dota 2 professional matches played in **Captain's Mode**, each game begins with a **draft phase** where two teams take turns banning and picking heroes. The draft follows a fixed sequence of 24 actions (bans and picks alternating in a set pattern). 

## Your Task

Build a model that predicts which hero will be **first-picked** in a given match.

- For each match, your model should output a **ranked list of hero candidates**.
- You are evaluated on **Accuracy@5**: the fraction of matches where the true first-picked hero appears in your model's top 5 predictions.
- **Validation set**: the last 200 matches in the dataset (by `start_time`).
- **Training set**: everything before the last 200 matches.
- You may only use data from this file — no external data sources.

## Time Budget

This task is designed for **2–3 hours** of focused work. A simple but well-validated baseline is better than a complex but broken solution.

---

## 1. Load the Dataset

In [ ]:
import json
import numpy as np
import pandas as pd

df = pd.read_csv("./data/test_task_dataset.csv")

print(f"Dataset: {len(df):,} matches, {df.shape[1]} columns")
print(f"Date range: {pd.to_datetime(df['start_time'].min(), unit='s')} to {pd.to_datetime(df['start_time'].max(), unit='s')}")
df.head(3)

Dataset: 92,685 matches, 13 columns
Date range: 2023-01-01 03:10:36 to 2026-01-31 18:51:45


,match_id,start_time,radiant_win,leagueid,cluster,radiant_team_id,dire_team_id,radiant_captain,dire_captain,series_id,series_type,picks_bans,first_pick_hero
0,6946804781,1672542636,True,14659,273,8629327.0,8629324.0,1090568489,230308002,737564.0,1.0,"[{""order"": 0, ""is_pick"": false, ""team"": 1, ""he...",5
1,6946831126,1672545026,True,14659,274,8629324.0,8629327.0,230308002,1090568489,737564.0,1.0,"[{""order"": 0, ""is_pick"": false, ""team"": 0, ""he...",5
2,6946866675,1672548081,True,14659,274,8629327.0,8629324.0,412910858,1194935883,737564.0,1.0,"[{""order"": 0, ""is_pick"": false, ""team"": 0, ""he...",121


---

## 2. Data Dictionary

Each row is one professional Dota 2 match played in Captain's Mode.

| Column | Type | Description |
|--------|------|-------------|
| `match_id` | int | Unique match identifier |
| `start_time` | int | Match start time as Unix timestamp (seconds since epoch).  |
| `radiant_win` | bool | `True` if Radiant won, `False` if Dire won. This is a post-match outcome |
| `leagueid` | int | Tournament/league identifier.  |
| `cluster` | int | Server cluster (region) where the match was played. |
| `radiant_team_id` | int | Registered team ID for Radiant side. May be missing for some matches. |
| `dire_team_id` | int | Registered team ID for Dire side. May be missing for some matches. |
| `radiant_captain` | int | Account ID of the Radiant team captain (the drafter). |
| `dire_captain` | int | Account ID of the Dire team captain (the drafter). |
| `series_id` | int | Series identifier  |
| `series_type` | int | `0` = Best-of-1, `1` = Best-of-3, `2` = Best-of-5. |
| `picks_bans` | str (JSON) | The full draft sequence — see detailed explanation below. |
| `first_pick_hero` | int | **TARGET** — the `hero_id` of the first picked hero in this match. |

### The `picks_bans` Column

This is a JSON array containing every action in the draft, sorted by `order`. Each action has:

| Field | Type | Meaning |
|-------|------|---------|
| `order` | int | Position in the draft sequence (0 = first action, 23 = last). |
| `is_pick` | bool | `true` = this hero was **picked**, `false` = this hero was **banned**. |
| `team` | int | `0` = Radiant, `1` = Dire. |
| `hero_id` | int | Numeric hero identifier. |


---

## 3. Parsing Example

Here's how to extract useful information from the `picks_bans` JSON.

In [4]:
def parse_picks_bans(picks_bans_str):
    """Returns dict with radiant_picks, dire_picks, radiant_bans, dire_bans (lists of hero_ids)."""
    actions = json.loads(picks_bans_str)
    result = {"radiant_picks": [], "dire_picks": [], "radiant_bans": [], "dire_bans": []}
    for a in sorted(actions, key=lambda x: x["order"]):
        team = "radiant" if a["team"] == 0 else "dire"
        kind = "picks" if a["is_pick"] else "bans"
        result[f"{team}_{kind}"].append(a["hero_id"])
    return result

# Example usage
parsed = parse_picks_bans(df.iloc[0]["picks_bans"])
print(f"Radiant picks: {parsed['radiant_picks']}")
print(f"Dire picks:    {parsed['dire_picks']}")
print(f"Radiant bans:  {parsed['radiant_bans']}")
print(f"Dire bans:     {parsed['dire_bans']}")
print(f"First pick:    hero_id {parsed['dire_picks'][0]}  (== first_pick_hero: {df.iloc[0]['first_pick_hero']})")

Radiant picks: [37, 32, 85, 44, 52]
Dire picks:    [5, 86, 97, 8, 14]
Radiant bans:  [137, 91, 9, 22, 53, 19, 35]
Dire bans:     [136, 84, 77, 25, 95, 36, 120]
First pick:    hero_id 5  (== first_pick_hero: 5)


---

## 4. Train / Validation Split

The dataset is sorted by `start_time`. Use the **last 200 matches** as your validation set.

In [5]:
VAL_SIZE = 200

train_df = df.iloc[:-VAL_SIZE].copy()
val_df = df.iloc[-VAL_SIZE:].copy()

print(f"Train: {len(train_df):,} matches  ({pd.to_datetime(train_df['start_time'].min(), unit='s').date()} to {pd.to_datetime(train_df['start_time'].max(), unit='s').date()})")
print(f"Val:   {len(val_df):,} matches  ({pd.to_datetime(val_df['start_time'].min(), unit='s').date()} to {pd.to_datetime(val_df['start_time'].max(), unit='s').date()})")

Train: 92,485 matches  (2023-01-01 to 2026-01-29)
Val:   200 matches  (2026-01-29 to 2026-01-31)


---

## 5. Evaluation Function

Your model must produce a function that, given a match row, returns a **list of hero_ids** ranked by predicted probability of being first-picked. We measure **Accuracy@5**: the fraction of validation matches where the true `first_pick_hero` is in your top 5.

In [6]:
def evaluate(val_df, predict_fn, k=5):
    """
    Evaluate accuracy@k on a validation set.

    Parameters
    ----------
    val_df : DataFrame
        Validation matches (each row has all columns from the dataset).
    predict_fn : callable
        Function that takes a single match row (pd.Series) and returns
        a list of hero_ids ranked by predicted likelihood of being first-picked.
        The list should have at least k elements.
    k : int
        Number of top predictions to consider.

    Returns
    -------
    float
        Accuracy@k (fraction of matches with true hero in top-k).
    """
    hits = 0
    for _, row in val_df.iterrows():
        top_k = predict_fn(row)[:k]
        if row["first_pick_hero"] in top_k:
            hits += 1
    accuracy = hits / len(val_df)
    print(f"Accuracy@{k}: {accuracy:.3f} ({hits}/{len(val_df)})")
    return accuracy

### Baseline: Most-Popular-Hero

As a sanity check, here's a trivial baseline that always predicts the globally most popular first-pick heroes.

In [ ]:
# Baseline: global first-pick popularity from training data
global_fp_ranking = train_df["first_pick_hero"].value_counts().index.tolist()

def predict_most_popular(match_row):
    return global_fp_ranking

print("Baseline (global popularity):")
evaluate(val_df, predict_most_popular, k=5)

Baseline (global popularity):
Accuracy@5: 0.110 (22/200)


0.11

---

## 6. Your Solution

Build your model below. Some ideas for features you might consider: